# Interactive Pipeline Run and Quick Charts

In [ ]:
import sys
import pathlib
import logging
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure repo root is on path (walk up until we find src/)
repo_root = pathlib.Path.cwd()
for _ in range(4):  # climb a few levels max
    if (repo_root / "src").exists():
        break
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.dq.anomaly.run_anomaly_detection import setup_logging, run_for_ecommerce, run_for_online_retail
from src.config.db_config import get_engine

logger = setup_logging("logs/notebook_run.log")

%matplotlib inline

## Run one dataset (adjust as needed)

In [ ]:
# Choose one at a time to keep interactive runs light
# run_for_ecommerce()
# run_for_online_retail()

## Anomaly metrics

In [ ]:
engine = get_engine()
metrics = pd.read_sql("select * from dq.anomaly_metrics order by created_at desc", engine)
metrics.head()

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=metrics, x="model_name", y="anomaly_rate", hue="source_table")
plt.ylabel("Anomaly rate")
plt.title("Anomaly rates by model")
plt.tight_layout()
plt.show()

## Validation issues

In [ ]:
issues = pd.read_sql(
    """
    select source_table, rule_name, count(*) as issues
    from dq.validation_issues
    group by source_table, rule_name
    """,
    engine,
)
issues.head()

In [ ]:
pivot = issues.pivot(index="rule_name", columns="source_table", values="issues").fillna(0)
plt.figure(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Blues")
plt.title("Validation issues by rule and table")
plt.tight_layout()
plt.show()

## Imputation audit

In [ ]:
audit = pd.read_sql("select * from audit.imputation_log order by created_at desc", engine)
audit.head()

## Before/after missingness (edit columns as needed)

In [ ]:
raw_counts = pd.read_sql(
    """
    select sum(case when price is null then 1 else 0 end) as price_nulls,
           sum(case when quantity is null then 1 else 0 end) as qty_nulls
    from raw.ecommerce_transactions
    """,
    engine,
)

proc_counts = pd.read_sql(
    """
    select sum(case when price is null then 1 else 0 end) as price_nulls,
           sum(case when quantity is null then 1 else 0 end) as qty_nulls
    from processed.ecommerce_transactions
    """,
    engine,
)

pd.DataFrame(
    [
        {"table": "raw", **raw_counts.iloc[0].to_dict()},
        {"table": "processed", **proc_counts.iloc[0].to_dict()},
    ]
)